# 5.1 Условная вероятность и теорема Байеса

Аналитика 101 · блок «Дата аналитика» · тема 05 «Вероятность» · страница 5.1.

В первой части вы смоделируете 100 000 визитов в интернет-магазин и проверку платежей. Во второй вычислите условные вероятности по учебной таблице `visits.csv` и сверите результат с формулой Байеса. В третьей проверите, есть ли связь между источником визита и покупкой.

**Как работать.** Начните с ячейки, которая находит папку с данными. Затем можно выполнить любой из трёх примеров.

Вывод получен на Python 3.13, NumPy 2.5, pandas 3.0 и SciPy 1.18.

## Где лежат данные

Скачайте общий набор данных главы 5. После распаковки папка `data/` и папка урока с ноутбуком должны лежать рядом. Первая ячейка найдёт файл `data/probability/visits.csv` и выберет рабочую папку для остальных ячеек.

In [1]:
import os
import pathlib
import platform

import numpy as np
import pandas as pd
import scipy

root = pathlib.Path.cwd()
while not (root / "data" / "probability").is_dir() and root != root.parent:
    root = root.parent                      # поднимаемся, пока рядом не окажется data/probability/
assert (root / "data" / "probability" / "visits.csv").exists(), \
    "Не найдена папка data/probability/. Скачайте общий набор темы 05 и положите data/ рядом с папкой страницы"
os.chdir(root)

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__)

Python 3.13.13 | numpy 2.5.0 | pandas 3.0.5 | scipy 1.18.1


## Оценка вероятности по случайным испытаниям

### Оценка условных вероятностей по 100 000 визитов

In [2]:
import numpy as np

rng = np.random.default_rng(51)
n = 100_000
# Доли источников и вероятность покупки задаём так же, как в генераторе visits.csv.
source = rng.choice(["органика", "реклама", "рассылка"], n, p=[0.55, 0.30, 0.15])
p_buy = np.select([source == "органика", source == "реклама", source == "рассылка"],
                  [0.025, 0.040, 0.060])
bought = rng.random(n) < p_buy          # Вероятность покупки зависит от источника визита.

ad = source == "реклама"
print("P(купил)           ", bought.mean().round(4))
print("P(купил | реклама) ", bought[ad].mean().round(4))   # доля внутри круга «реклама»
print("P(реклама | купил) ", ad[bought].mean().round(4))   # доля внутри круга «купил»

# Вычислим теоретические вероятности по формулам полной вероятности и Байеса.
p_b = 0.55 * 0.025 + 0.30 * 0.040 + 0.15 * 0.060
print("по формулам:        ", round(p_b, 4), 0.04, round(0.04 * 0.30 / p_b, 4))

# Теперь моделируем платежи. Правило отмечает 99% мошеннических операций и 1% обычных.
fraud = rng.random(n) < 0.01
alarm = np.where(fraud, rng.random(n) < 0.99, rng.random(n) < 0.01)
print("операций с предупреждением:", alarm.sum(), "из них мошеннических:", (fraud & alarm).sum())
print("P(фрод | сигнал)   ", fraud[alarm].mean().round(3))

P(купил)            0.0342
P(купил | реклама)  0.0394
P(реклама | купил)  0.3448
по формулам:         0.0348 0.04 0.3453
операций с предупреждением: 1938 из них мошеннических: 948
P(фрод | сигнал)    0.489


Параметры симуляции те же, что у учебной таблицы. Другое начальное значение генератора (51) даёт другую выборку. Для доли рекламы среди визитов с покупкой симуляция и формула дают 34,5% при округлении. В примере с платежами среди 1 938 отмеченных транзакций оказались мошенническими 948, или 48,9%. По формуле ожидаем 50%.

## Таблица сопряжённости и теорема Байеса

### Как выбрать знаменатель в pd.crosstab

In [3]:
import pandas as pd

v = pd.read_csv("data/probability/visits.csv")   # 100 000 учебных визитов: source, bought
t = pd.crosstab(v["source"], v["bought"], margins=True, margins_name="всего")
print(t)

# В столбце 1 видна конверсия каждого источника.
print(pd.crosstab(v["source"], v["bought"], normalize="index").round(4))
# В столбце 1 видна доля каждого источника среди визитов с покупкой.
print(pd.crosstab(v["source"], v["bought"], normalize="columns").round(4))

# Проверим формулу Байеса по той же таблице: P(реклама | купил) = P(купил | реклама) · P(реклама) / P(купил)
n = t.loc["всего", "всего"]
p_ad = t.loc["реклама", "всего"] / n
p_buy = t.loc["всего", 1] / n
p_buy_ad = t.loc["реклама", 1] / t.loc["реклама", "всего"]
print("Байес:", round(p_buy_ad * p_ad / p_buy, 4),
      "  прямо из столбца:", round(t.loc["реклама", 1] / t.loc["всего", 1], 4))

# полная вероятность: P(купил) = сумма P(купил | источник) · P(источник)
w = v["source"].value_counts(normalize=True)
conv = v.groupby("source")["bought"].mean()
print("полная вероятность:", round((conv * w).sum(), 5), " среднее bought:", v["bought"].mean())

bought        0     1   всего
source                       
органика  53802  1307   55109
рассылка  14146   897   15043
реклама   28676  1172   29848
всего     96624  3376  100000
bought         0       1
source                  
органика  0.9763  0.0237
рассылка  0.9404  0.0596
реклама   0.9607  0.0393
bought         0       1
source                  
органика  0.5568  0.3871
рассылка  0.1464  0.2657
реклама   0.2968  0.3472
Байес: 0.3472   прямо из столбца: 0.3472
полная вероятность: 0.03376  среднее bought: 0.03376


В строке «реклама» и столбце 1 доля по строке равна 0,0393. Это конверсия рекламы. Доля по столбцу равна 0,3472. Это доля рекламы среди визитов с покупкой. Теорема Байеса связывает эти две вероятности. По формуле полной вероятности получаем общую конверсию 0,03376.

## Проверка независимости

### Проверка независимости с помощью критерия хи-квадрат

In [4]:
import pandas as pd
from scipy.stats import chi2_contingency

v = pd.read_csv("data/probability/visits.csv")
t = pd.crosstab(v["source"], v["bought"])

# 1. Сравним доли визитов с покупкой по источникам с общей долей.
print(v.groupby("source")["bought"].mean().round(4).to_dict(), round(v["bought"].mean(), 4))

# 2. Число визитов, которое ожидается в каждой ячейке при независимости.
res = chi2_contingency(t)
print(pd.DataFrame(res.expected_freq, index=t.index, columns=t.columns).round(1))

# 3. Критерий хи-квадрат оценивает расхождение наблюдаемых и ожидаемых частот.
print(f"хи-квадрат = {res.statistic:.1f}, степеней свободы {res.dof}, p = {res.pvalue:.1e}")

# Для сравнения проверим, связана ли покупка с чётностью номера визита.
even = v["visit_id"] % 2 == 0
r2 = chi2_contingency(pd.crosstab(even, v["bought"]))
print(f"чётность: хи-квадрат = {r2.statistic:.2f}, p = {r2.pvalue:.2f}")

{'органика': 0.0237, 'рассылка': 0.0596, 'реклама': 0.0393} 0.0338
bought          0       1
source                   
органика  53248.5  1860.5
рассылка  14535.1   507.9
реклама   28840.3  1007.7
хи-квадрат = 506.8, степеней свободы 2, p = 9.1e-111
чётность: хи-квадрат = 0.33, p = 0.56


При независимости в ячейке «реклама, есть покупка» ожидалось бы 1 007,7 визита. В таблице их 1 172. Для рассылки ожидалось 507,9, а наблюдалось 897. По всей таблице статистика хи-квадрат равна 506,8 при двух степенях свободы, p ≈ 9,1 · 10⁻¹¹¹. При независимости расхождение такой величины или больше крайне маловероятно. Для чётности номера визита p = 0,56. На обычном уровне значимости 0,05 оснований отвергнуть независимость нет. Подробнее об этом в уроках 6.3 и 6.4.

## Что дальше

В папке урока есть скрипт `scene_points.py`. Он отбирает 1 000 визитов для диаграммы, сохраняя пропорции групп по источнику и покупке с точностью до округления. Запустите скрипт и сравните результат с долями, которые получаются при случайном отборе без учёта групп.